# 🧬 Módulo 4: Modelado de Proteínas y Docking Molecular
## Actividad 4.6: Docking Ligando-Proteína con AutoDock Vina

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_04_modelado_proteinas_docking/06_docking_ligando_proteina.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Preparar ligandos con RDKit
- Definir el sitio activo (grid box)
- Ejecutar docking con AutoDock Vina
- Analizar poses de unión
- Evaluar energías de interacción
- Visualizar resultados con Py3Dmol

---

## 📚 Introducción

En esta actividad aplicarás docking molecular para predecir cómo se une un ligando pequeño al sitio activo de una proteína.

---

In [ ]:
# Instalación de dependencias
!pip install rdkit-pypi py3Dmol biopython requests numpy pandas matplotlib 2>/dev/null || \
  pip install rdkit py3Dmol biopython requests numpy pandas matplotlib
# AutoDock Vina Python bindings (vina >= 1.2.3)
!pip install vina 2>/dev/null || echo "Instala vina con: conda install -c conda-forge vina"
print("✓ Dependencias instaladas")

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from pathlib import Path
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw, Descriptors
    from rdkit.Chem import rdMolDescriptors
    from rdkit.Chem.Draw import IPythonConsole
    print("✓ RDKit disponible")
except ImportError:
    print("⚠️  RDKit no disponible — instala con: conda install -c conda-forge rdkit")

print("✓ Importaciones completadas")

## 📚 Caso de Estudio: COX-2 + Celecoxib

Trabajaremos con un sistema clásico en diseño de fármacos:
- **Receptor**: Ciclooxigenasa-2 (COX-2) — enzima relacionada con inflamación y dolor
- **Ligando**: Celecoxib (Celebrex®) — AINE inhibidor selectivo de COX-2

Esta combinación es ideal para aprender docking porque:
- La estructura cristalográfica del complejo está disponible en PDB (**3LN1**)
- Permite **validar** los resultados comparando con la pose experimental

## 1. Preparación del Receptor

In [ ]:
Path("docking_cox2").mkdir(exist_ok=True)

def descargar_pdb(pdb_id, output_dir="docking_cox2"):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url)
    out = Path(output_dir) / f"{pdb_id}.pdb"
    if r.status_code == 200:
        out.write_text(r.text)
        print(f"✓ {pdb_id}.pdb descargado ({out.stat().st_size//1024} KB)")
        return out
    print(f"✗ Error descargando {pdb_id}")
    return None

class SelectorProteinaA(Select):
    """Conserva sólo residuos proteicos de la cadena A."""
    def accept_chain(self, chain):
        return chain.id == 'A'
    def accept_residue(self, residue):
        return residue.id[0] == ' '  # Sin agua ni heteroátomos

def preparar_receptor(pdb_id="3LN1", output_dir="docking_cox2"):
    """Descarga y prepara el receptor COX-2 para docking."""
    pdb_raw = descargar_pdb(pdb_id, output_dir)
    if not pdb_raw:
        return None
    
    # Limpiar: sólo cadena A proteica
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("cox2", pdb_raw)
    
    pdb_limpio = Path(output_dir) / f"{pdb_id}_receptor.pdb"
    io = PDBIO()
    io.set_structure(struct)
    io.save(str(pdb_limpio), SelectorProteinaA())
    
    # Contar residuos
    struct2 = parser.get_structure("limpio", pdb_limpio)
    n_res = sum(1 for ch in struct2[0] for r in ch if r.id[0]==' ')
    print(f"✓ Receptor preparado: {n_res} residuos → {pdb_limpio}")
    return pdb_limpio

receptor = preparar_receptor("3LN1")

## 2. Preparación del Ligando

El ligando debe estar en formato adecuado:
- **3D optimizado**: geometría tridimensional con carga apropiada
- **Formato .pdbqt**: el que acepta AutoDock Vina (incluye cargas parciales)

### Celecoxib
- SMILES: `Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1`
- Peso molecular: 381.4 g/mol
- Inhibidor selectivo COX-2, Ki = 40 nM

In [ ]:
def preparar_ligando_rdkit(smiles, nombre="ligando", output_dir="docking_cox2"):
    """
    Prepara un ligando desde SMILES:
    1. Genera conformación 3D con MMFF94
    2. Guarda como SDF
    3. Convierte a PDBQT (si Open Babel disponible)
    """
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem, Descriptors
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"✗ SMILES inválido: {smiles}")
            return None
        
        # Agregar hidrógenos y generar conformación 3D
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)
        
        # Propiedades calculadas
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        rotbonds = Descriptors.NumRotatableBonds(mol)
        
        print(f"\n{'='*40}")
        print(f"  LIGANDO: {nombre}")
        print(f"{'='*40}")
        print(f"  MW:         {mw:.2f} g/mol")
        print(f"  LogP:       {logp:.2f}")
        print(f"  H-Donors:   {hbd}")
        print(f"  H-Acceptors:{hba}")
        print(f"  Rot. bonds: {rotbonds}")
        
        # Verificar Regla de Lipinski
        lipinski_ok = (mw < 500 and logp < 5 and hbd <= 5 and hba <= 10)
        print(f"  Lipinski:   {'✓ CUMPLE' if lipinski_ok else '✗ No cumple'}")
        
        # Guardar SDF
        out_sdf = Path(output_dir) / f"{nombre}.sdf"
        writer = Chem.SDWriter(str(out_sdf))
        writer.write(mol)
        writer.close()
        print(f"\n  SDF guardado: {out_sdf}")
        
        return mol, out_sdf
        
    except ImportError:
        print("⚠️  RDKit no disponible. Instala con: conda install -c conda-forge rdkit")
        return None, None

# Preparar Celecoxib
smiles_celecoxib = "Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1"
resultado = preparar_ligando_rdkit(smiles_celecoxib, "celecoxib")

## 3. Definición del Sitio Activo (Grid Box)

El sitio activo de COX-2 se define a partir de:
1. Las coordenadas del ligando co-cristalizado en la estructura 3LN1
2. Residuos clave del sitio activo: Arg120, Tyr355, Ser530, His90

### Coordenadas del sitio activo de COX-2 (3LN1)
Estas coordenadas se determinan extrayendo el centroide del ligando co-cristalizado.

In [ ]:
def obtener_centroide_ligando(pdb_file, resname="CEL"):
    """Extrae las coordenadas del ligando co-cristalizado para definir el grid box."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("prot", pdb_file)
    
    coords = []
    for model in struct:
        for chain in model:
            for residue in chain:
                if residue.resname.strip() == resname:
                    for atom in residue:
                        coords.append(atom.coord)
    
    if not coords:
        print(f"⚠️  Ligando '{resname}' no encontrado. Usando coordenadas predeterminadas.")
        # Coordenadas del sitio activo de COX-2 (conocidas de literatura)
        return np.array([22.0, 5.0, 18.0])
    
    centroide = np.mean(coords, axis=0)
    print(f"✓ Centroide del ligando {resname}:")
    print(f"  x = {centroide[0]:.3f}")
    print(f"  y = {centroide[1]:.3f}")
    print(f"  z = {centroide[2]:.3f}")
    return centroide

def generar_config_vina(receptor_pdbqt, ligando_pdbqt, centro, 
                         tamano=(22,22,22), exhaustividad=8, output_dir="docking_cox2"):
    """Genera el archivo de configuración para AutoDock Vina."""
    config = f"""# Configuración de AutoDock Vina
# Receptor
receptor = {receptor_pdbqt}

# Ligando
ligand = {ligando_pdbqt}

# Sitio de unión (Grid Box)
center_x = {centro[0]:.3f}
center_y = {centro[1]:.3f}
center_z = {centro[2]:.3f}

size_x = {tamano[0]}
size_y = {tamano[1]}
size_z = {tamano[2]}

# Parámetros de búsqueda
exhaustiveness = {exhaustividad}
num_modes = 9
energy_range = 3

# Salida
out = {output_dir}/docking_resultado.pdbqt
log = {output_dir}/docking_log.txt
"""
    config_file = Path(output_dir) / "vina_config.txt"
    config_file.write_text(config)
    print(f"✓ Configuración Vina guardada en: {config_file}")
    print(f"\n{'='*45}")
    print(config)
    return config_file

# Obtener centro del sitio activo desde el PDB original
if receptor:
    pdb_raw = Path("docking_cox2") / "3LN1.pdb"
    centro = obtener_centroide_ligando(str(pdb_raw), resname="CEL")
    config = generar_config_vina(
        receptor_pdbqt="3LN1_receptor.pdbqt",
        ligando_pdbqt="celecoxib.pdbqt",
        centro=centro,
        output_dir="docking_cox2"
    )

## 4. Ejecución del Docking con AutoDock Vina

### Opción A: Vina Python API (vina >= 1.2.3)
Permite ejecutar Vina directamente desde Python.

### Opción B: Línea de comandos
Si tienes Vina instalado como ejecutable:
```bash
vina --config vina_config.txt
```

### Conversión a PDBQT con Open Babel
Antes de ejecutar Vina, los archivos deben estar en formato `.pdbqt`:
```bash
# Receptor
obabel receptor.pdb -O receptor.pdbqt -p 7.4 --partialcharge gasteiger -xr
# Ligando
obabel ligando.sdf -O ligando.pdbqt --partialcharge gasteiger -gen3d
```

In [ ]:
def ejecutar_docking_vina(receptor_pdbqt, ligando_pdbqt, centro, tamano=(22,22,22),
                          exhaustividad=8, n_poses=9):
    """
    Ejecuta AutoDock Vina usando la API de Python.
    Requiere: pip install vina (y AutoDock Vina >= 1.2.3)
    """
    try:
        from vina import Vina
        
        v = Vina(sf_name='vina')
        v.set_receptor(receptor_pdbqt)
        v.set_ligand_from_file(ligando_pdbqt)
        v.compute_vina_maps(
            center=list(centro),
            box_size=list(tamano)
        )
        
        print("Ejecutando docking (puede tomar 1-5 minutos)...")
        v.dock(exhaustiveness=exhaustividad, n_poses=n_poses)
        
        poses = v.poses(n_poses=n_poses)
        energias = v.energies(n_poses=n_poses)
        
        print("\n✓ Docking completado")
        print(f"\n{'Modo':>5} | {'ΔG (kcal/mol)':>15}")
        print("-" * 25)
        for i, e in enumerate(energias[:, 0], 1):
            print(f"  {i:3d}  | {e:12.2f}")
        
        return poses, energias
        
    except ImportError:
        print("⚠️  Módulo 'vina' no instalado.")
        print("   Opción 1: pip install vina")
        print("   Opción 2: conda install -c conda-forge vina")
        print("\n   Usando resultados simulados para demostración...")
        return simular_docking_cox2()
    
    except Exception as e:
        print(f"Error en docking: {e}")
        print("Usando resultados simulados...")
        return simular_docking_cox2()

def simular_docking_cox2():
    """
    Simula resultados típicos de docking COX-2 + Celecoxib.
    Basado en valores reportados en la literatura.
    """
    # Energías típicas reportadas para celecoxib en COX-2
    energias = np.array([
        [-10.2, 0, 0],   # Pose 1 (mejor)
        [-9.8,  1.2, 2.1],
        [-9.5,  1.8, 2.9],
        [-9.1,  3.4, 4.2],
        [-8.9,  3.7, 5.1],
        [-8.6,  4.2, 5.8],
        [-8.3,  5.1, 6.4],
        [-8.1,  5.8, 7.2],
        [-7.9,  6.3, 7.8]
    ])
    
    print("📊 Resultados simulados — COX-2 + Celecoxib")
    print(f"{'Modo':>5} | {'ΔG (kcal/mol)':>15} | {'RMSD lb':>8} | {'RMSD ub':>8}")
    print("-" * 45)
    for i, row in enumerate(energias, 1):
        marca = " ← MEJOR" if i == 1 else ""
        print(f"  {i:3d}  | {row[0]:12.1f}   | {row[1]:8.3f} | {row[2]:8.3f}{marca}")
    
    print(f"\nEnergía de unión (pose 1): {energias[0,0]:.1f} kcal/mol")
    print(f"Equivalente a Kd ≈ {np.exp(energias[0,0]/(1.987e-3*298.15))*1e9:.1f} nM")
    return None, energias

# Ejecutar docking
poses, energias = ejecutar_docking_vina(
    receptor_pdbqt="docking_cox2/3LN1_receptor.pdbqt",
    ligando_pdbqt="docking_cox2/celecoxib.pdbqt",
    centro=np.array([22.0, 5.0, 18.0])  # Sitio activo COX-2
)

## 5. Análisis de Resultados

In [ ]:
def analizar_resultados_docking(energias):
    """Genera análisis gráfico completo de los resultados de docking."""
    if energias is None:
        print("No hay resultados para analizar.")
        return
    
    scores = energias[:, 0]
    rmsd_lb = energias[:, 1]
    rmsd_ub = energias[:, 2]
    n_poses = len(scores)
    poses_num = np.arange(1, n_poses + 1)
    
    fig = plt.figure(figsize=(15, 10))
    
    # --- Panel 1: Scores por pose ---
    ax1 = fig.add_subplot(2, 3, 1)
    colores = ['gold' if s == min(scores) else 'steelblue' for s in scores]
    bars = ax1.bar(poses_num, scores, color=colores, edgecolor='white', linewidth=1.2)
    ax1.axhline(-9, color='green', linestyle='--', alpha=0.7, label='Límite buen fármaco')
    ax1.set_xlabel('Pose')
    ax1.set_ylabel('ΔG (kcal/mol)')
    ax1.set_title('Energías por Pose')
    ax1.legend(fontsize=8)
    ax1.set_xticks(poses_num)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Anotar mejor pose
    ax1.annotate(f'Mejor:\n{min(scores):.1f}', 
                xy=(poses_num[0], min(scores)),
                xytext=(poses_num[0]+1, min(scores)+0.5),
                fontsize=8, color='darkgoldenrod',
                arrowprops=dict(arrowstyle='->', color='darkgoldenrod'))
    
    # --- Panel 2: Score vs RMSD ---
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.scatter(rmsd_lb[1:], scores[1:], c='steelblue', s=80, alpha=0.8, label='Poses 2-N')
    ax2.scatter(rmsd_lb[0], scores[0], c='gold', s=200, marker='*', zorder=5, label='Pose 1')
    ax2.axvline(2.0, color='red', linestyle='--', alpha=0.6, label='RMSD = 2 Å')
    ax2.set_xlabel('RMSD lb (Å)')
    ax2.set_ylabel('ΔG (kcal/mol)')
    ax2.set_title('Score vs RMSD')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # --- Panel 3: Distribución de scores ---
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.hist(scores, bins=max(3, n_poses//2), color='steelblue', 
             edgecolor='white', alpha=0.8)
    ax3.axvline(np.mean(scores), color='red', linestyle='--', 
               label=f'Media: {np.mean(scores):.1f}')
    ax3.set_xlabel('ΔG (kcal/mol)')
    ax3.set_ylabel('Frecuencia')
    ax3.set_title('Distribución de Scores')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    
    # --- Panel 4: Resumen estadístico ---
    ax4 = fig.add_subplot(2, 1, 2)
    ax4.axis('off')
    
    R = 1.987e-3  # kcal/(mol·K)
    kd_mejor = np.exp(min(scores) / (R * 298.15))
    
    resumen = [
        ['Parámetro', 'Valor', 'Interpretación'],
        ['Mejor score (ΔG)', f'{min(scores):.2f} kcal/mol', 
         'Excelente' if min(scores) < -9 else 'Bueno' if min(scores) < -7 else 'Moderado'],
        ['Kd estimada', f'{kd_mejor*1e9:.2f} nM' if kd_mejor < 1e-6 else f'{kd_mejor*1e6:.2f} μM', ''],
        ['RMSD lb pose 2', f'{rmsd_lb[1]:.2f} Å', 
         'Pose similar' if rmsd_lb[1] < 2 else 'Pose diferente'],
        ['Poses con ΔG < -9', f'{sum(s < -9 for s in scores)}/{n_poses}', ''],
        ['Rango energético', f'{max(scores)-min(scores):.1f} kcal/mol', 
         'Amplio' if max(scores)-min(scores) > 3 else 'Estrecho'],
    ]
    
    tabla = ax4.table(
        cellText=resumen[1:],
        colLabels=resumen[0],
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    tabla.auto_set_font_size(True)
    tabla.set_fontsize(10)
    for (row, col), cell in tabla.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2196F3')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f5f5f5')
    
    ax4.set_title('Resumen del Análisis de Docking', fontsize=12, fontweight='bold', pad=10)
    
    plt.suptitle('COX-2 + Celecoxib — Análisis de Docking', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

analizar_resultados_docking(energias)

## 6. Visualización del Complejo

### Interacciones proteína-ligando importantes en COX-2

Los residuos clave que interactúan con celecoxib en COX-2:
- **Arg120**: Puente de hidrógeno con el sulfonamido
- **Tyr355**: Puente de hidrógeno
- **Ser530**: Contacto hidrofóbico
- **Val523, Leu384, Tyr385**: Pocket hidrofóbico
- **His90**: Coordinación con el grupo sulfonamido

In [ ]:
def visualizar_complejo_docking(pdb_file, resname_ligando="CEL",
                                residuos_sitio_activo=None):
    """
    Visualiza el complejo proteína-ligando con Py3Dmol.
    Resalta el ligando y los residuos del sitio activo.
    """
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=900, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Proteína: cartoon semitransparente
    view.setStyle({'chain': 'A', 'hetflag': False},
                 {'cartoon': {'color': 'lightblue', 'opacity': 0.8}})
    
    # Ligando: sticks llamativos
    view.setStyle({'resn': resname_ligando},
                 {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.25}})
    view.addSurface(py3Dmol.VDW, 
                   {'opacity': 0.4, 'colorscheme': 'greenCarbon'},
                   {'resn': resname_ligando})
    
    # Residuos del sitio activo
    if residuos_sitio_activo:
        for res_id in residuos_sitio_activo:
            view.addStyle({'resi': res_id, 'chain': 'A'},
                         {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.2}})
    
    view.zoomTo({'resn': resname_ligando})
    view.setBackgroundColor('white')
    return view

# Visualizar complejo COX-2 + celecoxib experimental (desde 3LN1)
pdb_3ln1 = Path("docking_cox2") / "3LN1.pdb"
if pdb_3ln1.exists():
    view = visualizar_complejo_docking(
        str(pdb_3ln1),
        resname_ligando="CEL",
        residuos_sitio_activo=[120, 355, 530, 90, 523, 384, 385]
    )
    print("Naranja: residuos del sitio activo | Verde: celecoxib experimental")
    view.show()
else:
    print("Descarga primero el PDB 3LN1 ejecutando la celda anterior.")

## 7. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Prepara el ligando **ibuprofeno** para docking:
- SMILES: `CC(C)Cc1ccc(cc1)C(C)C(=O)O`
- Calcula sus propiedades fisicoquímicas
- ¿Cumple la regla de Lipinski?

### Ejercicio 2 (Intermedio)
Con la proteína **CDK2** (PDB: 1HCL) y el ligando staurosporina:
- SMILES: `C[C@@H]1OC2=CC3=C(C=C2[C@H]1NC(=O)c1ccccc1)C1=C(N3)C=CC=C1N`
1. Prepara el receptor (cadena A, sin heteroátomos)
2. Identifica el sitio activo extrayendo el ligando co-cristalizado (STA)
3. Genera el archivo de configuración de Vina

### Ejercicio 3 (Avanzado)
Realiza un **análisis comparativo** de 3 inhibidores de COX-2:
- Celecoxib: `Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1`
- Rofecoxib: `CS(=O)(=O)c1ccc(-c2cc(=O)oc2)cc1` 
- Diclofenaco: `OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl`
1. Calcula las propiedades fisicoquímicas de cada uno
2. ¿Cuál predice mejor afinidad según el modelo?

In [ ]:
# Ejercicio 1: Ibuprofeno
smiles_ibuprofeno = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
resultado_ibu = preparar_ligando_rdkit(smiles_ibuprofeno, "ibuprofeno")

# Tu código aquí para los ejercicios 2 y 3...

## 8. Referencias

1. Trott, O. & Olson, A.J. (2010). AutoDock Vina: improving the speed and accuracy of docking. *J. Comput. Chem.*, 31, 455-461.
2. Wang, J. et al. (2004). Development and testing of a general Amber force field. *J. Comput. Chem.*, 25, 1157-1174.
3. Xu, S. et al. (2019). Celecoxib inhibits growth and induces apoptosis in cervical cancer. *Am. J. Cancer Res.*
4. Bhatt, D.L. et al. (2019). COX-2 inhibitors. *N. Engl. J. Med.*
5. Eberhardt, J. et al. (2021). AutoDock Vina 1.2.0: New docking methods, expanded force field, and python bindings. *J. Chem. Inf. Model.*

---

## 📚 Recursos Adicionales

### Software
- [AutoDock Vina](http://vina.scripps.edu/) — Descarga y documentación oficial
- [AutoDockTools 1.5.7](http://mgltools.scripps.edu/) — GUI para preparar archivos PDBQT
- [PyRx](https://pyrx.sourceforge.io/) — Interfaz gráfica con Vina integrado
- [Open Babel](https://openbabel.org/) — Conversión de formatos moleculares
- [ProLIF](https://prolif.readthedocs.io/) — Análisis de interacciones proteína-ligando

### Servidores web (sin instalación)
- [SwissDock](http://www.swissdock.ch/) — Docking online gratuito
- [DockingServer](https://www.dockingserver.com/) — Docking con interfaz web

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Preparar el receptor proteico para docking (limpieza, PDB → PDBQT)
- ✅ Preparar el ligando desde SMILES con RDKit (3D + propiedades)
- ✅ Definir el grid box usando el ligando co-cristalizado
- ✅ Ejecutar AutoDock Vina (API Python o línea de comandos)
- ✅ Interpretar los resultados: scores, RMSD, clustering de poses
- ✅ Visualizar el complejo proteína-ligando con Py3Dmol

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 4.6: Docking Ligando-Proteína**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_4.5-Fundamentos_de_Docking-blue.svg)](05_docking_fundamentos.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_4.7_➡️-Docking_Proteína_Proteína-green.svg)](07_docking_proteina_proteina.ipynb)

---

📚 **[Volver al Módulo 4](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>